# **Atividade Prática**
<font size=3>

- **Tema:** Transformer + BERT + LoRA.
- **Prazo de entrega:** 06 de Julho.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Enunciado:**
<font size=3>

Esta atividade está dividida em duas parte:
- **a.** Treinar um **modelo Transformer** para **traduação** do **idioma-fonte** para o **idioma-alvo**;
- **b.** Usar o **_encoder_** do Transformer **pré-treinado** para realizar uma tarefa de **regressão** *ou* **classificação**.
  

### **Parte _b_**:
<font size=3>

- Escolha um *dataset* para realizar uma tarefa de classificação ou regressão, com base no **idioma-fonte**;
- Utilize o *encoder* pré-treinado para construir o modelo BERT: anexando o *encoder* a uma camada *pooler*, seguida de uma camada de saída;
- Utilize a abordagem da LoRA para realizar um fine-tuning do BERT;
- Realize a avaliação do modelo e redija sua análise.


## **Resolução:**

### Imports e variaveis globais

In [1]:
import tensorflow as tf
import os
import json

import random
import zipfile

import pandas as pd
import requests
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model, Sequential
from transformers import BertTokenizerFast

In [2]:

MODEL_EXPORT_DIR = "atividade5-modelos"

TRANSLATION_MODEL_EXPORT_PATH = os.path.join(
    MODEL_EXPORT_DIR,
    "transformer_en_pt_savedmodel"
)

ENCODER_MODEL_EXPORT_PATH = os.path.join(
    MODEL_EXPORT_DIR,
    "encoder_en_savedmodel"
)

TRANSLATION_WEIGHTS_PATH = os.path.join(
    MODEL_EXPORT_DIR,
    "transformer_en_pt.weights.h5"
)

ENCODER_WEIGHTS_PATH = os.path.join(
    MODEL_EXPORT_DIR,
    "encoder_en.weights.h5"
)

RANDOM_STATE = 42

MAX_LEN_SOURCE = 64

D_MODEL = 128
NUM_HEADS = 4
DENSE_DIM = 512
NUM_ENCODER_LAYERS = 2
DROPOUT_RATE = 0.1

tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-cased"
)

VOCAB_SIZE = tokenizer.vocab_size

IMDB_DATASET_PATH = "./dataset/imdb.csv"

TEST_SIZE = 0.10
VALIDATION_SIZE = 0.10

MAX_LEN_CLASSIFICATION = MAX_LEN_SOURCE

CLASSIFICATION_BATCH_SIZE = 32

LORA_RANK = 8
LORA_ALPHA = 16

POOLER_UNITS = D_MODEL
CLASSIFIER_DROPOUT_RATE = 0.3
CLASSIFICATION_LEARNING_RATE = 1e-3


In [3]:
print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices("GPU"))
#tf.debugging.set_log_device_placement(True)

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Preparando MultiHeadAttention

Copiando `MultiHeadAttention` do arquivo `myattention.py`.

In [6]:
class MultiHeadAttention(layers.Layer):
    
    def __init__(self, d_model, num_heads, dropout_rate=0.1, return_weights=False):
        super().__init__()

        self.h = num_heads
        self.d_m = d_model
        
        assert d_model % num_heads == 0 # d_m deve ser divisível pelo número de cabeças
        
        self.d_k = d_model//num_heads

        self.scaler = tf.math.sqrt(tf.cast(self.d_k, tf.float32))

        self.return_weights = return_weights
        
        # camadas lineares para projeção (q·W^Q, k·W^K, v·W^V):
        self.proj_q = layers.Dense(d_model)
        self.proj_k = layers.Dense(d_model)
        self.proj_v = layers.Dense(d_model)
        
        # camada linear da saída (W^O):
        self.proj_o = layers.Dense(d_model)
        
        # camada dropout para os pesos de atenção:
        self.dropout = layers.Dropout(dropout_rate)

        # avisando ao Keras que esta classe suporta máscaras:
        self.supports_masking = True
        
        
    def split_heads(self, x, batch_size):
        """
        Divide a última dimensão em (h, d_k).
        Transpõe o resultado para shape (batch_size, h, L, d_k)
        """
        
        x = tf.reshape(x, (batch_size, -1, self.h, self.d_k))
        
        return tf.transpose(x, perm=[0, 2, 1, 3])

    
    def attention(self, Q, K, V, mask=None, training=False):
        
        scores = tf.matmul(Q, K, transpose_b=True)/self.scaler
    
        # Mask: se houver máscara, somamos um número muito negativo (-1e9)
        # para que a softmax zere essas posições.
        if mask is not None:
            # a máscara entra como (batch, L), mas "scores" é (batch, h, L, L), 
            # logo, faremos mask -> (batch, 1, 1, L)    
            if len(mask.shape) == 2:
                mask = mask[:, tf.newaxis, tf.newaxis, :]
                
            mask = tf.cast(mask, dtype=tf.float32)
            scores += (1 - mask)*(-1e9)
    
        weights = tf.nn.softmax(scores, axis=-1)
        
        weights = self.dropout(weights, training=training)
    
        output = tf.matmul(weights, V)

        return output, weights

    
    def call(self, q, k, v, mask=None, training=False):
        
        batch_size = tf.shape(q)[0]
        
        # projeções lineares:
        Q = self.proj_q(q)   # (batch_size, L, d_m)
        K = self.proj_k(k)   # (batch_size, L, d_m)
        V = self.proj_v(v)   # (batch_size, L, d_m)
        
        # dividindo em múltiplas cabeças:
        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        # calculando em paralelo todas as cabeças de atenção: 
        scaled_attention, attention_weights = self.attention(Q, K, V, mask, training=training)
        
        # reshape da "concatenação:"
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_m))
        
        # projeção final:
        output = self.proj_o(concat_attention)

        if self.return_weights: return output, attention_weights
        else: return output

### Copia das classes `TokenAndPositionEmbedding`, `EncoderLayerTraducao`, `EncoderLayerTraducao` da atividade 5a

In [7]:
class TokenAndPositionEmbedding(layers.Layer):
    """
    Camada de embedding de tokens + posições.
    """
    
    def __init__(
        self,
        vocab_size,
        max_len,
        d_model,
        dropout_rate=0.1,
        name=None
    ):
        super().__init__(name=name)

        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model

        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=d_model,
            name="token_embedding"
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=d_model,
            name="position_embedding"
        )

        self.dropout = layers.Dropout(
            dropout_rate
        )

    def call(self, input_ids, training=False):
        seq_len = tf.shape(input_ids)[1]

        positions = tf.range(
            start=0,
            limit=seq_len,
            delta=1
        )

        token_embeddings = self.token_embedding(
            input_ids
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = token_embeddings + position_embeddings

        return self.dropout(
            x,
            training=training
        )

In [8]:
class EncoderLayerTraducao(layers.Layer):
    """
    Camada encoder do Transformer.
    """
    
    def __init__(
        self,
        d_model,
        num_heads,
        dense_dim,
        dropout_rate=0.1,
        name=None
    ):
        super().__init__(name=name)

        self.self_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout_rate=dropout_rate
        )

        self.ffn = Sequential(
            [
                layers.Dense(
                    dense_dim,
                    activation="gelu"
                ),
                layers.Dense(
                    d_model
                )
            ],
            name="ffn"
        )

        self.norm1 = layers.LayerNormalization(
            epsilon=1e-12
        )

        self.norm2 = layers.LayerNormalization(
            epsilon=1e-12
        )

        self.dropout = layers.Dropout(
            dropout_rate
        )

    def call(
        self,
        x,
        padding_mask,
        training=False
    ):
        attention_output = self.self_attention(
            x,
            x,
            x,
            mask=padding_mask,
            training=training
        )

        x = self.norm1(
            x + attention_output
        )

        ffn_output = self.ffn(
            x
        )

        ffn_output = self.dropout(
            ffn_output,
            training=training
        )

        x = self.norm2(
            x + ffn_output
        )

        return x

In [9]:
class EncoderTraducao(Model):
    """
    Encoder isolado do Transformer treinado na Parte 5a.
    """
    
    def __init__(
        self,
        vocab_size,
        max_len,
        d_model,
        num_heads,
        dense_dim,
        num_layers,
        dropout_rate=0.1,
        name="encoder_en"
    ):
        super().__init__(
            name=name
        )

        self.encoder_embedding = TokenAndPositionEmbedding(
            vocab_size=vocab_size,
            max_len=max_len,
            d_model=d_model,
            dropout_rate=dropout_rate,
            name="encoder_embedding"
        )

        self.encoder_layers = [
            EncoderLayerTraducao(
                d_model=d_model,
                num_heads=num_heads,
                dense_dim=dense_dim,
                dropout_rate=dropout_rate,
                name=f"encoder_layer_{i + 1}"
            )
            for i in range(num_layers)
        ]

    def call(
        self,
        inputs,
        training=False
    ):
        encoder_input_ids = inputs["encoder_input_ids"]
        encoder_padding_mask = inputs["encoder_padding_mask"]

        encoder_output = self.encoder_embedding(
            encoder_input_ids,
            training=training
        )

        for encoder_layer in self.encoder_layers:
            encoder_output = encoder_layer(
                encoder_output,
                padding_mask=encoder_padding_mask,
                training=training
            )

        return encoder_output

### Carregando o Encoder pré-treinado

In [ ]:
encoder_model = EncoderTraducao(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN_SOURCE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dense_dim=DENSE_DIM,
    num_layers=NUM_ENCODER_LAYERS,
    dropout_rate=DROPOUT_RATE,
    name="encoder_en"
)

entrada_teste_encoder = {
    "encoder_input_ids": tf.zeros(
        shape=(1, MAX_LEN_SOURCE),
        dtype=tf.int32
    ),
    "encoder_padding_mask": tf.ones(
        shape=(1, MAX_LEN_SOURCE),
        dtype=tf.int32
    )
}

_ = encoder_model(
    entrada_teste_encoder,
    training=False
)

encoder_model.load_weights(
    ENCODER_WEIGHTS_PATH
)

saida_teste_encoder = encoder_model(
    entrada_teste_encoder,
    training=False
)

print("Encoder carregado com sucesso.")
print("Entrada:", entrada_teste_encoder["encoder_input_ids"].shape)
print("Saída:", saida_teste_encoder.shape)
print("Pesos carregados de:", ENCODER_WEIGHTS_PATH)

2026-07-06 21:51:13.110798: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Max
2026-07-06 21:51:13.110827: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-07-06 21:51:13.110836: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1783385473.110852 48186727 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1783385473.110877 48186727 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Encoder carregado com sucesso.
Entrada: (1, 64)
Saída: (1, 64, 128)
Pesos carregados de: atividade5-modelos/encoder_en.weights.h5


### Atividade de Classificação

Nesta etapa, decidimos revisitar a tarefa de classificação de sentimentos feita na Atividade 2 da Unidade 3. A ideia é usar novamente o dataset local do IMDB, porque assim conseguimos comparar este novo experimento com um resultado que já conhecemos.

Na Atividade 2, o modelo treinado especificamente para classificação de sentimentos teve um desempenho bem forte no conjunto de teste, com acurácia de `0,8994`, F1-score de `0,9005` e ROC-AUC de `0,9629`. Esses valores servem como uma boa referência para avaliar até onde conseguimos chegar agora.

A diferença desta vez é que não vamos treinar um modelo de classificação começando do zero. Vamos reaproveitar o encoder treinado na etapa anterior, em que trabalhamos com tradução de inglês para português, e verificar se ele também consegue ajudar em uma tarefa diferente: identificar se uma review é positiva ou negativa.

### Lendo dataset IMBD e divisão

In [11]:
def carregar_dataset_imdb_local():
    """
    Carrega o dataset local IMDB utilizado nas atividades anteriores.

    O dataset possui:
    - reviews: texto da review;
    - label: classe binária do sentimento.
    """
    df = (
        pd.read_csv(IMDB_DATASET_PATH)
        .sample(
            frac=1,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

    return df


def dividir_dataset_classificacao(df):
    """
    Divide o dataset em treino, validação e teste.

    Primeiro separa 10% para teste, como na Atividade 2.
    Depois separa 10% do conjunto de treino para validação.
    Todas as divisões preservam a proporção das classes.
    """
    X = df["reviews"]
    y = df["label"]

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train_full
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


def comparar_distribuicao_classes_classificacao(
    y_total,
    y_train,
    y_val,
    y_test
):
    """
    Exibe a distribuição das classes nos conjuntos de treino, validação e teste.
    """
    distribuicao = pd.DataFrame({
        "Total - qtd": y_total.value_counts().sort_index(),
        "Total - %": y_total.value_counts(normalize=True).sort_index() * 100,
        "Treino - qtd": y_train.value_counts().sort_index(),
        "Treino - %": y_train.value_counts(normalize=True).sort_index() * 100,
        "Validação - qtd": y_val.value_counts().sort_index(),
        "Validação - %": y_val.value_counts(normalize=True).sort_index() * 100,
        "Teste - qtd": y_test.value_counts().sort_index(),
        "Teste - %": y_test.value_counts(normalize=True).sort_index() * 100
    })

    distribuicao.index.name = "Classe"

    return distribuicao.style.format({
        "Total - qtd": "{:,.0f}",
        "Total - %": "{:.2f}%",
        "Treino - qtd": "{:,.0f}",
        "Treino - %": "{:.2f}%",
        "Validação - qtd": "{:,.0f}",
        "Validação - %": "{:.2f}%",
        "Teste - qtd": "{:,.0f}",
        "Teste - %": "{:.2f}%"
    })


df_imdb = carregar_dataset_imdb_local()

X_train_classification, X_val_classification, X_test_classification, y_train_classification, y_val_classification, y_test_classification = (
    dividir_dataset_classificacao(
        df_imdb
    )
)

print("Dataset completo:", df_imdb.shape)
print("Treino:", X_train_classification.shape, y_train_classification.shape)
print("Validação:", X_val_classification.shape, y_val_classification.shape)
print("Teste:", X_test_classification.shape, y_test_classification.shape)

display(
    comparar_distribuicao_classes_classificacao(
        y_total=df_imdb["label"],
        y_train=y_train_classification,
        y_val=y_val_classification,
        y_test=y_test_classification
    )
)

display(
    df_imdb.head()
)

Dataset completo: (50000, 2)
Treino: (40500,) (40500,)
Validação: (4500,) (4500,)
Teste: (5000,) (5000,)


,Total - qtd,Total - %,Treino - qtd,Treino - %,Validação - qtd,Validação - %,Teste - qtd,Teste - %
Classe,,,,,,,,
0,"25,000",50.00%,"20,250",50.00%,"2,250",50.00%,"2,500",50.00%
1,"25,000",50.00%,"20,250",50.00%,"2,250",50.00%,"2,500",50.00%


,reviews,label
0,River's Edge is more than just the story of a ...,1
1,Evening is the beautiful story of the flawed l...,1
2,I question anyone saying they don't care for t...,1
3,This docu-drama is what you would expect from ...,1
4,"As several posters have ""hinted,"" this is a so...",0


### Tokenização Wordpiece

In [12]:
# =============================================================================
# Parte 5b - Tokenização WordPiece para classificação
# =============================================================================

def tokenizar_textos_classificacao(textos):
    """
    Tokeniza os textos de entrada para o encoder pré-treinado.

    O tokenizer adiciona automaticamente [CLS] no início e [SEP] no final.
    """
    tokenizado = tokenizer(
        list(textos.astype(str)),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN_CLASSIFICATION,
        return_attention_mask=True
    )

    input_ids = tf.constant(
        tokenizado["input_ids"],
        dtype=tf.int32
    )

    attention_mask = tf.constant(
        tokenizado["attention_mask"],
        dtype=tf.int32
    )

    return input_ids, attention_mask


def preparar_arrays_classificacao(
    X,
    y
):
    """
    Prepara entradas e rótulos para a tarefa de classificação.
    """
    input_ids, attention_mask = tokenizar_textos_classificacao(
        X
    )

    entradas = {
        "encoder_input_ids": input_ids,
        "encoder_padding_mask": attention_mask
    }

    labels = tf.constant(
        y.to_numpy(),
        dtype=tf.float32
    )

    return entradas, labels


def criar_dataset_classificacao(
    X,
    y,
    embaralhar=False
):
    """
    Cria um tf.data.Dataset para classificação de sentimentos.
    """
    entradas, labels = preparar_arrays_classificacao(
        X,
        y
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            entradas,
            labels
        )
    )

    if embaralhar:
        dataset = dataset.shuffle(
            buffer_size=len(y),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(
        CLASSIFICATION_BATCH_SIZE
    )

    return dataset


train_classification_dataset = criar_dataset_classificacao(
    X_train_classification,
    y_train_classification,
    embaralhar=True
)

validation_classification_dataset = criar_dataset_classificacao(
    X_val_classification,
    y_val_classification,
    embaralhar=False
)

test_classification_dataset = criar_dataset_classificacao(
    X_test_classification,
    y_test_classification,
    embaralhar=False
)


for entradas_batch, labels_batch in train_classification_dataset.take(1):
    print("encoder_input_ids:", entradas_batch["encoder_input_ids"].shape)
    print("encoder_padding_mask:", entradas_batch["encoder_padding_mask"].shape)
    print("labels:", labels_batch.shape)

    print()
    print("Exemplo tokenizado:")
    print(
        tokenizer.convert_ids_to_tokens(
            entradas_batch["encoder_input_ids"][0].numpy()
        )
    )

    print()
    print("Label:", labels_batch[0].numpy())

encoder_input_ids: (32, 64)
encoder_padding_mask: (32, 64)
labels: (32,)

Exemplo tokenizado:
['[CLS]', 'See', 'No', 'Evil', 'is', 'the', 'first', 'film', 'from', 'WWE', 'films', '.', 'Yes', 'WWE', ',', 'Word', 'Wrestling', 'Entertainment', ',', 'pro', 'wrestling', '.', 'Of', 'course', 'being', 'that', 'it', "'", 's', 'a', 'WWE', 'film', 'a', 'wrestler', 'has', 'to', 'star', 'in', 'it', ',', 'the', 'wrestler', 'being', 'Glenn', 'Jacobs', 'aka', 'Kane', '.', 'Which', 'is', 'not', 'really', 'important', 'as', 'if', 'you', 'didn', "'", 't', 'know', 'Kane', 'or', 'what', '[SEP]']

Label: 1.0


2026-07-06 21:51:28.997838: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Classificador

In [ ]:
def congelar_encoder(
    encoder
):
    """
    Congela todos os pesos do encoder pré-treinado.
    """
    encoder.trainable = False

    for layer in encoder.layers:
        layer.trainable = False


class DenseLoRA(layers.Layer):
    """
    Camada Dense com adaptação LoRA.
    """

    def __init__(
        self,
        units,
        rank=8,
        alpha=16,
        activation=None,
        name=None
    ):
        super().__init__(
            name=name
        )

        self.units = units
        self.rank = rank
        self.alpha = alpha

        self.activation = tf.keras.activations.get(
            activation
        )

        self.base_dense = layers.Dense(
            units=units,
            activation=None,
            trainable=False,
            name="base_dense"
        )

    def build(
        self,
        input_shape
    ):
        input_dim = int(
            input_shape[-1]
        )

        self.base_dense.build(
            input_shape
        )

        self.lora_a = self.add_weight(
            name="lora_a",
            shape=(input_dim, self.rank),
            initializer=tf.keras.initializers.RandomNormal(
                stddev=0.01
            ),
            trainable=True
        )

        self.lora_b = self.add_weight(
            name="lora_b",
            shape=(self.rank, self.units),
            initializer="zeros",
            trainable=True
        )

        super().build(
            input_shape
        )

    def call(
        self,
        inputs
    ):
        base_output = self.base_dense(
            inputs
        )

        lora_output = tf.matmul(
            tf.matmul(
                inputs,
                self.lora_a
            ),
            self.lora_b
        )

        output = base_output + (
            lora_output * (self.alpha / self.rank)
        )

        if self.activation is not None:
            output = self.activation(
                output
            )

        return output


def construir_modelo_classificacao_lora(
    encoder
):
    """
    Constrói o classificador binário com encoder congelado e LoRA no pooler.
    """
    congelar_encoder(
        encoder
    )

    encoder_input_ids = layers.Input(
        shape=(MAX_LEN_CLASSIFICATION,),
        dtype=tf.int32,
        name="encoder_input_ids"
    )

    encoder_padding_mask = layers.Input(
        shape=(MAX_LEN_CLASSIFICATION,),
        dtype=tf.int32,
        name="encoder_padding_mask"
    )

    encoder_output = encoder(
        {
            "encoder_input_ids": encoder_input_ids,
            "encoder_padding_mask": encoder_padding_mask
        },
        training=False
    )

    cls_output = layers.Lambda(
        lambda x: x[:, 0, :],
        name="cls_token"
    )(
        encoder_output
    )

    x = DenseLoRA(
        units=POOLER_UNITS,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
        activation="tanh",
        name="pooler_lora"
    )(
        cls_output
    )

    x = layers.Dropout(
        CLASSIFIER_DROPOUT_RATE,
        name="dropout"
    )(
        x
    )

    output = layers.Dense(
        units=1,
        activation="sigmoid",
        name="output"
    )(
        x
    )

    model = Model(
        inputs={
            "encoder_input_ids": encoder_input_ids,
            "encoder_padding_mask": encoder_padding_mask
        },
        outputs=output,
        name="bert_encoder_imdb_lora"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=CLASSIFICATION_LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    return model


classification_model = construir_modelo_classificacao_lora(
    encoder_model
)

classification_model.summary()

Model: "bert_encoder_imdb_lora"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input_ids   │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_padding_ma… │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_en          │ (None, 64, 128)   │  4,116,224 │ encoder_input_id… │
│ (EncoderTraducao)   │                   │            │ encoder_padding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_token (Lambda)  │ (None, 128)       │          0 │ encoder_en[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pooler_lora         │ (None, 128)       │     18,560 │ cls_token[0][0]   │
│ (DenseLoRA)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ pooler_lora[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,134,913 (15.77 MB)

 Trainable params: 2,177 (8.50 KB)

 Non-trainable params: 4,132,736 (15.77 MB)

### Treinamento

In [ ]:
CLASSIFICATION_EPOCHS = 20
CLASSIFICATION_EARLY_STOPPING_PATIENCE = 3

classification_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=CLASSIFICATION_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    )
]

classification_history = classification_model.fit(
    train_classification_dataset,
    validation_data=validation_classification_dataset,
    epochs=CLASSIFICATION_EPOCHS,
    callbacks=classification_callbacks
)

Epoch 1/20


2026-07-06 21:54:04.455596: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1266/1266 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.5122 - auc: 0.5170 - loss: 0.7063 - val_accuracy: 0.5347 - val_auc: 0.5717 - val_loss: 0.6886
Epoch 2/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.5350 - auc: 0.5462 - loss: 0.6908 - val_accuracy: 0.5393 - val_auc: 0.5901 - val_loss: 0.6855
Epoch 3/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.5351 - auc: 0.5518 - loss: 0.6894 - val_accuracy: 0.5651 - val_auc: 0.5889 - val_loss: 0.6825
Epoch 4/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.5451 - auc: 0.5616 - loss: 0.6874 - val_accuracy: 0.5393 - val_auc: 0.5927 - val_loss: 0.6861
Epoch 5/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.5447 - auc: 0.5639 - loss: 0.6870 - val_accuracy: 0.5509 - val_auc: 0.5946 - val_loss: 0.6842
Epoch 6/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.5464 - auc: 0.5656 - loss: 0.6867 - val_accuracy: 0.5480 - val_auc: 0.6010 - val_loss: 0.6834


In [ ]:
classification_test_results = classification_model.evaluate(
    test_classification_dataset
)

print("Resultado no conjunto de teste:")
for metric_name, metric_value in zip(
    classification_model.metrics_names,
    classification_test_results
):
    print(f"{metric_name}: {metric_value:.6f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5816 - auc: 0.6048 - loss: 0.6806
Resultado no conjunto de teste:
loss: 0.680558
compile_metrics: 0.581600


### Conclusão

O resultado obtido mostra que o reaproveitamento do encoder treinado na tarefa de tradução da atividade 5a funcionou tecnicamente, mas teve desempenho limitado na classificação de sentimentos. O modelo obteve `loss = 0.680558`, `accuracy = 0.581600` e AUC próxima de `0.6048` no conjunto de teste. Como a tarefa é binária e o dataset é balanceado, uma acurácia de 50% representaria desempenho aleatório; portanto, o modelo ficou acima do acaso, mas ainda distante dos resultados obtidos na atididade 2.

Considerando que o encoder foi mantido congelado e apenas uma pequena adaptação LoRA com a cabeça de classificação foi treinada, totalizando somente `2.177` parâmetros treináveis de um total de aproximadamente `4,13 milhões`, esse comportamento é esperado. Isso restringe bastante a capacidade de adaptação do modelo à nova tarefa.

Outro fator importante é o truncamento das reviews. Na atividade 2, o dataset IMDB foi analisado considerando reviews longas, com uma janela de contexto de 581 tokens. Aqui nessa atividade, entretanto, foi necessário limitar as entradas a `64` tokens, pois o encoder da primeira parte foi treinado com embedding desse tamanho. Com isso, boa parte do conteúdo das reviews é descartada, reduzindo a informação disponível para a classificação.

Portanto, o experimento buscou cumprir o objetivo principal da atividade, que era reutilizar o encoder pré-treinado em uma nova tarefa supervisionada com adaptação via LoRA. O resultado indica que houve algum aprendizado transferido, mas também evidencia que o encoder treinado para tradução e congelado não produz, por si só, representações suficientemente fortes para análise de sentimentos em reviews longas.